# Trace Count v19: v18 with Shared Decimal Digit Tokenization

This notebook isolates one change from v18: **every ordinal index and
final count is rendered with the same ten decimal digit tokens**.
Data distributions, length 1024, count range 1-128, ten marker types,
model architecture, optimizer, and the four-run
`uniform/power x direct/CoT` comparison stay fixed.

Direct completion is `<Count> digits(n) <NumEnd>`. CoT completion is
`<Index> digits(1) M1 ... <Index> digits(n) Mn </Think>
<Count> digits(n) <NumEnd>`. For example, index 12 is three semantic
pieces: `<Index>`, digit `1`, digit `2`; the final digit `2` is the
attention query that predicts marker 12. The final scalar uses the
same digits but is introduced by `<Count>`.

**Main/all is four independent 10,000-step runs.** Checkpoints sync to
Drive every 2,000 steps. All reported behavior is free-running greedy
generation, including multi-digit parsing and the explicit number-end
delimiter.


In [ ]:
from pathlib import Path

IN_COLAB = False
DRIVE_READY = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
    DRIVE_READY = True
except ImportError:
    print("Local runtime: Google Drive mount skipped.")

DRIVE_RESULTS_ROOT = Path(
    "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/"
    "Synthetic_CoT_NiaH_Count/colab_results"
)
if DRIVE_READY:
    DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print({"in_colab": IN_COLAB, "drive_ready": DRIVE_READY})


## 1. Repository and environment


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
if IN_COLAB:
    ROOT = Path("/content/Synthetic_CoT_NiaH_Count")
    if not (ROOT / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent]
    ROOT = next((path.resolve() for path in candidates if (path / "pyproject.toml").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("Run this notebook from the repository or use Colab.")

os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print("repo =", ROOT)
print("python =", sys.executable)


In [ ]:
def stream_command(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(" ".join(map(str, command)), flush=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        captured = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
            captured.append(line.rstrip())
        returncode = process.wait()
    if returncode:
        print("---- Last 160 log lines ----")
        print("\n".join(captured[-160:]))
        raise subprocess.CalledProcessError(returncode, command)


## 2. Configuration and four-run suite manifest


In [ ]:
import torch
from synthetic_counting_v19.config import canonical_run_specs, preset_config, select_specs

PRESET = "main"  # use "debug" for a quick end-to-end check
SUITE = "all"    # "power", "uniform", or "all"
# all = train + attention + state + plots. Existing checkpoints can
# be analyzed with "attention", "state", or "plots" without retraining.
STAGE = "all"
SEED = 1234
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_ROOT = "runs/synthetic_counting_v19"
RUN_NAME = f"v19_{PRESET}_{SUITE}_seed{SEED}"
SKIP_COMPLETED = True
CHECKPOINT_SYNC_ROOT = DRIVE_RESULTS_ROOT / "v19_live_checkpoints" if DRIVE_READY else None

CONFIG = preset_config(PRESET, seed=SEED, device=DEVICE)
SPECS = select_specs(SUITE, canonical_run_specs()) if PRESET == "main" else ()
print(CONFIG.to_dict())
print({"suite": SUITE, "number_of_training_runs": len(SPECS) if SPECS else 2})
if PRESET == "main" and SUITE == "all":
    print("This launches 4 x 10,000-step runs: uniform/power x direct/CoT.")


In [ ]:
import pandas as pd
from IPython.display import display

if SPECS:
    display(pd.DataFrame([spec.__dict__ for spec in SPECS]))


## 3. Run the focused suite


In [ ]:
cmd = [
    sys.executable, "-u", "-m", "synthetic_counting_v19.run_v19",
    "--preset", PRESET,
    "--suite", SUITE,
    "--stage", STAGE,
    "--device", DEVICE,
    "--seed", str(SEED),
    "--out-root", OUT_ROOT,
    "--run-name", RUN_NAME,
]
if SKIP_COMPLETED:
    cmd.append("--skip-completed")
if CHECKPOINT_SYNC_ROOT is not None:
    cmd += ["--checkpoint-sync-root", str(CHECKPOINT_SYNC_ROOT)]
stream_command(cmd, Path(OUT_ROOT) / "last_pipeline.log")
RUN_DIR = Path(OUT_ROOT) / RUN_NAME
assert (RUN_DIR / "config.json").exists(), RUN_DIR
print("RUN_DIR =", RUN_DIR.resolve())


## 4. Definitions and result tables

**Digit grammar.** `D0..D9` are shared by trace indices and the
final answer. `<Index>` and `<Count>` identify the number's role;
`<NumEnd>` terminates the final scalar. Leading zeros, zero, and
values outside the configured range are invalid. Thus exact
final-count accuracy requires every generated digit and the
delimiter to parse to the gold integer.

**Free-running metrics.** `primary_accuracy` and
`token_accuracy` compare the parsed final integer with gold.
`enumeration_accuracy` requires a well-formed sequence of
indices `1..n` followed by `</Think>`. `trace_marker_accuracy`
measures position-wise marker identity, and
`trace_exact_accuracy` requires the complete digit-indexed trace.

**Attention anchors.** For trace step `k`, the query that predicts
marker `M_k` is the final decimal digit of `k`, not `<Index>` and
not necessarily its first digit. `correct_prompt_needle_mass` is
raw attention from that query to prompt needle `k`.
`diagonal_dominance` normalizes that mass within the prompt-needle
subset, while `correct_top1` asks whether it is the strongest
needle. At marker `M_k`, `next_prompt_needle_mass` probes successor
preparation. The final-answer attention/state anchor is the
`<Count>` role token, whose next-token logits start the scalar.

**State scores.** Hidden-state index 0 is the embedding; indices
1-4 are residual states after Layers 1-4. Held-out nearest-
centroid accuracy and ridge `R^2` measure exact and approximately
linear count/progress readability. PCA is fitted to label
centroids and exported through PC1-PC6.


In [ ]:
from IPython.display import Image, display

for filename in [
    "suite_manifest.csv",
    "final_summary.csv",
    "final_by_band.csv",
    "final_by_count.csv",
    "dynamics_summary.csv",
    "dynamics_by_band.csv",
    "attention_summary.csv",
    "state_probe_summary.csv",
    "state_pca_variance.csv",
]:
    path = RUN_DIR / "tables" / filename
    if path.exists() and path.stat().st_size:
        print("\n", filename)
        display(pd.read_csv(path))
for path in sorted((RUN_DIR / "figures").glob("*.png")):
    print(path.name)
    display(Image(filename=str(path)))


## 5. Interactive PC1-PC6 count-centroid geometry


In [ ]:
import itertools
import ipywidgets as widgets
import plotly.express as px

centroids = pd.read_csv(RUN_DIR / "tables" / "state_centroids_pca.csv")
pc_columns = [column for column in centroids.columns if column.startswith("pc")]
axis_options = list(itertools.combinations(pc_columns, 3))
run_widget = widgets.Dropdown(options=sorted(centroids["run_name"].unique()), description="run")
site_widget = widgets.Dropdown(options=sorted(centroids["site"].unique()), description="site")
layer_widget = widgets.Dropdown(options=sorted(centroids["layer"].unique()), description="state")
axes_widget = widgets.Dropdown(options=[(" / ".join(x), x) for x in axis_options], description="PC axes")

def show_geometry(run_name, site, layer, pc_axes):
    subset = centroids[
        (centroids["run_name"] == run_name)
        & (centroids["site"] == site)
        & (centroids["layer"] == layer)
    ].sort_values("state_label")
    if subset.empty:
        print("No rows for this run/site/state-index combination.")
        return
    x, y, z = pc_axes
    figure = px.scatter_3d(
        subset, x=x, y=y, z=z, color="state_label",
        hover_data=["state_label", "mode", "distribution"],
        title=f"{run_name} | {site} | hidden-state index {layer}",
        color_continuous_scale="Viridis",
    )
    figure.update_traces(marker={"size": 4})
    figure.show()

controls = {"run_name": run_widget, "site": site_widget, "layer": layer_widget, "pc_axes": axes_widget}
display(widgets.HBox([run_widget, site_widget, layer_widget, axes_widget]))
display(widgets.interactive_output(show_geometry, controls))


## 6. Save the complete result bundle to Google Drive


In [ ]:
import json
import shutil
from datetime import datetime

DRIVE_SAVE_COMPLETED = False
if DRIVE_READY:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    destination = DRIVE_RESULTS_ROOT / f"{RUN_NAME}_{stamp}"
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(RUN_DIR, destination)
    DRIVE_SAVE_COMPLETED = True
    print("Saved result bundle:", destination)
else:
    print("Drive is unavailable; results remain at", RUN_DIR)


## 7. Optional GitHub step


In [ ]:
# Optional. Keep disabled unless you intentionally want to commit notebook/code changes.
PUSH_TO_GITHUB = False
if PUSH_TO_GITHUB:
    subprocess.run(["git", "status", "--short"], check=True)
    print("Review the status above, then commit and push intentionally from a terminal.")


## 8. Disconnect only after a verified Drive save


In [ ]:
AUTO_DISCONNECT_AFTER_DRIVE_SAVE = True
if IN_COLAB and AUTO_DISCONNECT_AFTER_DRIVE_SAVE and DRIVE_SAVE_COMPLETED:
    from google.colab import runtime
    print("Drive save verified; disconnecting the Colab runtime.")
    runtime.unassign()
else:
    print("Runtime kept alive (not Colab, save incomplete, or auto-disconnect disabled).")
